# 13 · ClearML pipeline + experiment tracking

Wire the PoC stages as ClearML Tasks. Run locally first (`start_locally`). Remote queues need a ClearML Agent, which this laptop compose stack does not start.

In [ ]:
from cross_model_drift.config import load_config

config = load_config()
config.clearml_project, config.clearml_web_host, config.clearml_api_host

## Suggested tasks

`prepare_data` → `create_features` → `train_logistic_baseline` → `train_lightgbm` → `run_optuna_hpo` → `evaluate_model` → `compare_champion_challenger`

In [ ]:
from clearml import PipelineController

from cross_model_drift.data import load_split
from cross_model_drift.features import target_vector
from cross_model_drift.metrics import quality_metrics
from cross_model_drift.models import train_lightgbm


def prepare_data(version: str, split: str) -> int:
    frame = load_split(version, split)  # type: ignore[arg-type]
    return len(frame)


def train_and_evaluate(version: str) -> dict[str, float]:
    train = load_split(version, "train")  # type: ignore[arg-type]
    valid = load_split(version, "validation")  # type: ignore[arg-type]
    test = load_split(version, "test")  # type: ignore[arg-type]
    model = train_lightgbm(train, target_vector(train), valid, target_vector(valid))
    return quality_metrics(target_vector(test), model.predict_proba(test))


pipe = PipelineController(
    name="fraud-champion-challenger",
    project=config.clearml_project,
    version="0.1.0",
    add_pipeline_tags=False,
)
pipe.add_function_step(
    name="prepare_v1_train",
    function=prepare_data,
    function_kwargs={"version": "v1", "split": "train"},
    function_return=["n_rows"],
)
pipe.add_function_step(
    name="train_v1",
    function=train_and_evaluate,
    function_kwargs={"version": "v1"},
    function_return=["v1_metrics"],
    parents=["prepare_v1_train"],
)
pipe.add_function_step(
    name="train_v2",
    function=train_and_evaluate,
    function_kwargs={"version": "v2"},
    function_return=["v2_metrics"],
    parents=["prepare_v1_train"],
)

# After ~/.clearml/clearml.conf exists:
# pipe.start_locally(run_pipeline_steps_locally=False)
"pipeline defined; start locally once credentials exist"

Open http://localhost:8080 after compose is up. Create API credentials, write `~/.clearml/clearml.conf` from `configs/clearml.conf.example`, then set `init=True` in earlier notebooks.